<a href="https://colab.research.google.com/github/sakkarchanda/Minor-Project/blob/master/minorphase2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
model.save_pretrained("/content/drive/MyDrive/harmful_model")
tokenizer.save_pretrained("/content/drive/MyDrive/harmful_model")

NameError: name 'model' is not defined

In [5]:
!zip -r harmful_model.zip harmful_model

	zip warning: name not matched: harmful_model

zip error: Nothing to do! (try: zip -r harmful_model.zip . -i harmful_model)


In [4]:
!zip -r harmful_model.zip harmful_model

	zip warning: name not matched: harmful_model

zip error: Nothing to do! (try: zip -r harmful_model.zip . -i harmful_model)


In [3]:
from google.colab import files
files.download("harmful_model.zip")

FileNotFoundError: Cannot find file: harmful_model.zip

In [ ]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from transformers import Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# ---------------- LOAD DATA ----------------
harmful = pd.read_csv("harmful.csv")
harmless = pd.read_csv("harmless.csv")

# Fix spelling mistake
harmful.rename(columns={"deacription": "description"}, inplace=True)
harmless.rename(columns={"deacription": "description"}, inplace=True)

harmful["label"] = 1
harmless["label"] = 0

df = pd.concat([harmful, harmless], ignore_index=True)

df["text"] = (
    df["title"].fillna("") + " " +
    df["description"].fillna("") + " " +
    df["transcript"].fillna("")
)

# ---------------- SPLIT ----------------
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["text"].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

# ---------------- TOKENIZER ----------------
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=256)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=256)

class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = CustomDataset(train_encodings, train_labels)
test_dataset = CustomDataset(test_encodings, test_labels)

# ---------------- MODEL ----------------
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

# ---------------- METRICS ----------------
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

# ---------------- TRAINING ARGS (v5 SAFE) ----------------
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_steps=100,
    save_strategy="epoch"
)

# ---------------- TRAINER ----------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

metrics = trainer.evaluate()
print("\nFinal Evaluation Metrics:")
print(metrics)

model.save_pretrained("harmful_model")
tokenizer.save_pretrained("harmful_model")

print("\nModel training completed successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
100,0.645594
200,0.542204
300,0.526669
400,0.486918
500,0.488763
600,0.480043
700,0.460064
800,0.459177
900,0.465896
1000,0.365491


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Final Evaluation Metrics:
{'eval_loss': 0.6390056014060974, 'eval_accuracy': 0.7982527982527983, 'eval_f1': 0.8234169653524492, 'eval_precision': 0.8259827420901247, 'eval_recall': 0.820867079561696, 'eval_runtime': 26.9952, 'eval_samples_per_second': 135.691, 'eval_steps_per_second': 8.483, 'epoch': 3.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model training completed successfully!


In [9]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from transformers import Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# ---------------- LOAD BAD WORDS ----------------
with open("en.txt", encoding="utf-8") as f:
    bad_words = [w.strip().lower() for w in f if w.strip()]

def count_bad_words(text):
    text = text.lower()
    return sum(word in text for word in bad_words)

# ---------------- LOAD DATA ----------------
harmful = pd.read_csv("harmful.csv")
harmless = pd.read_csv("harmless.csv")

# Fix column typo
harmful.rename(columns={"deacription": "description"}, inplace=True)
harmless.rename(columns={"deacription": "description"}, inplace=True)

harmful["label"] = 1
harmless["label"] = 0

df = pd.concat([harmful, harmless], ignore_index=True)

# ---------------- CREATE TEXT FIELD ----------------
df["text"] = (
    df["title"].fillna("") + " " +
    df["description"].fillna("") + " " +
    df["transcript"].fillna("")
)

# ---------------- ADD BAD WORD FEATURE ----------------
df["badword_count"] = df["text"].apply(count_bad_words)

# Add as prefix token
df["text"] = "BADCOUNT_" + df["badword_count"].astype(str) + " " + df["text"]

# ---------------- SPLIT ----------------
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["text"].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

# ---------------- TOKENIZER ----------------
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=256)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=256)

# ---------------- DATASET CLASS ----------------
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = CustomDataset(train_encodings, train_labels)
test_dataset = CustomDataset(test_encodings, test_labels)

# ---------------- MODEL ----------------
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

# ---------------- METRICS ----------------
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

# ---------------- TRAINING ARGS ----------------
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_steps=100,
    save_strategy="epoch"
)

# ---------------- TRAINER ----------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

metrics = trainer.evaluate()
print("\nFinal Evaluation Metrics:")
print(metrics)

# ---------------- SAVE MODEL ----------------
model.save_pretrained("harmful_model")
tokenizer.save_pretrained("harmful_model")

print("\nModel training completed successfully with Bad Word Integration!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Final Evaluation Metrics:
{'eval_loss': 0.6426231265068054, 'eval_accuracy': 0.800982800982801, 'eval_f1': 0.8263045032165832, 'eval_precision': 0.8265014299332698, 'eval_recall': 0.8261076703191996, 'eval_runtime': 28.8426, 'eval_samples_per_second': 127.0, 'eval_steps_per_second': 7.94, 'epoch': 3.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model training completed successfully with Bad Word Integration!


In [10]:
# Save model + tokenizer
model.save_pretrained("harmful_model")
tokenizer.save_pretrained("harmful_model")

print("Model saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully!


In [13]:
from transformers import DistilBertTokenizer
DistilBertTokenizer.from_pretrained("distilbert-base-uncased").save_pretrained("harmful_model")

('harmful_model/tokenizer_config.json', 'harmful_model/tokenizer.json')

In [14]:
!zip -r harmful_model.zip harmful_model

  adding: harmful_model/ (stored 0%)
  adding: harmful_model/config.json (deflated 49%)
  adding: harmful_model/tokenizer_config.json (deflated 42%)
  adding: harmful_model/model.safetensors (deflated 8%)
  adding: harmful_model/tokenizer.json (deflated 71%)


In [19]:
from google.colab import files
files.download("harmful_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
from google.colab import files
files.download("harmful_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>